In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis_scripts.graphs import set_size
from analysis_scripts.load_lammps import load_lammps

from pizza.dump import dump

plt.style.use('seaborn')
plt.style.use('tex')
dumpsdir = os.path.abspath("../dumps/")

In [ ]:
nsmc = 15
testname = "test1_prob1_rate5000_cutoff2_nsmc" + str(nsmc)
testype = "parallel_unfix"


bonds_d = dump(os.path.join(dumpsdir, f"{testname}_bond_{testype}.lammpstrj"))
coord_d = dump(os.path.join(dumpsdir, f"{testname}_dump_{testype}.lammpstrj"))
coord_d.sort()
try:
    coord_d.unwrap()
except Exception:
    print("Already unwrapped")


In [ ]:
bonds = bonds_d.all_atoms_info()[:, :, 1:]
bonds = np.concatenate(
    (np.repeat(np.full(bonds.shape[0], 1).cumsum(), bonds.shape[1]).reshape(
        bonds.shape[0], bonds.shape[1], 1
    ),
    bonds),
    axis=2,
)
itype = coord_d.all_atoms_info()[:, :, :5]
itype = np.concatenate(
    (np.repeat(np.full(itype.shape[0], 1).cumsum(), itype.shape[1]).reshape(
        itype.shape[0], itype.shape[1], 1
    ),
    itype),
    axis=2,
)

In [ ]:
itype2 = itype[(itype[:,:,2]==2)]
bonds2 = bonds[(bonds[:,:,3]==2)]
bonds3 = bonds[(bonds[:,:,3]==3)]

typedf = pd.DataFrame(itype2, columns=["time","id","type"])
bonds2df = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])
bonds3df = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])

if np.any(bonds2df[["time","type"]].groupby("time").count()>nsmc):
    print("Detected multiple bond")
if np.any(bonds3df[["time","type"]].groupby("time").count()>nsmc):
    print("Detected multiple bond")
if np.any(typedf[["time","type"]].groupby("time").count()>2*nsmc):
    print("Detected multiple type assegnation")

bonds2 = np.sort(bonds2.reshape(-1,nsmc,4),axis=1)
bonds3 = np.sort(bonds3.reshape(-1,nsmc,4),axis=1)
itype2 = np.sort(itype2.reshape(-1,nsmc,2,3),axis=1)

In [ ]:
plt.figure(figsize=set_size(500))
ax = plt.gca()

ax.set_xlabel("Time [It]")
ax.set_ylabel("Indexes")

for i in range(nsmc):
    ax.fill_between(bonds2[:,i,0],bonds2[:,i,2],bonds2[:,i,1], alpha =0.5, color="grey")
    ax.fill_between(bonds3[:,i,0],bonds3[:,i,2],bonds3[:,i,1], alpha =0.5, color="brown")

    ax.scatter(itype2[:,i,0,0],itype2[:,i,0,1], s=0.5,c="blue",alpha=0.6)
    ax.scatter(itype2[:,i,0,0],itype2[:,i,1,1], s=0.5,c="red",alpha=0.6)

plt.savefig(f"results/kymograph_{testname}_{testype}.pdf")